# 05 — GPU Training (pilot) — Colab

Run in a **Colab GPU runtime**. Shows GPU specs, runs a synthetic pilot
through the *same* shared engine with `device=cuda`.

For the controlled Experiment D, run this notebook twice in the **same
runtime**: once with `device=cpu`, once with `device=cuda` — everything
else identical.

In [ ]:
import sys, time, json, platform
from pathlib import Path
root = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(root / "src"))

In [ ]:
import torch
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print("VRAM (MB):", round(props.total_memory/1e6, 1))
    print("SM count:", props.multi_processor_count)
    print("CUDA:", torch.version.cuda)
    print("Device capability:", torch.cuda.get_device_capability(0))
else:
    print("No GPU — switch the Colab runtime (Runtime > Change runtime type).")

In [ ]:
# Experiment D device switch — change ONLY this line between runs
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("using device:", DEVICE)

In [ ]:
from roadvision.config import load_config, config_hash
from roadvision.models.detector import build_detector, make_synthetic_batch
from roadvision.engine.train import build_optimizer, train_one_epoch

config = load_config(root / "configs" / "experiment.yaml")
print("config hash (must match the CPU run of Experiment D):", config_hash(config))
device = torch.device(DEVICE)
model = build_detector(config.to_dict())
# Experiment D: snapshot initial weights BEFORE any update (EXPERIMENT_DESIGN §1.1)
init_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
model.to(device)          # engine never moves the model implicitly
opt = build_optimizer(model, config.to_dict())

class SynthLoader:
    batch_size = 4
    def __iter__(self):
        for _ in range(5):
            yield make_synthetic_batch(batch_size=4, seed=0)

# Runtime warm-up (kernels/allocator) — EXCLUDED from reported stats.
# Warm-up changes weights, so the initial state is RESTORED afterwards
# (state-restore warm-up, EXPERIMENT_DESIGN §4) before the measured epoch.
for _ in range(3):
    imgs, tgts = make_synthetic_batch(batch_size=4, seed=1)
    if device.type == "cuda":
        imgs = [i.to(device) for i in imgs]
        tgts = [{k: v.to(device) for k, v in t.items()} for t in tgts]
    loss = sum(model(imgs, tgts).values())
    opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
if device.type == "cuda":
    torch.cuda.synchronize()
model.load_state_dict(init_state)              # discard warm-up weight drift
opt = build_optimizer(model, config.to_dict()) # fresh optimizer = initial state

report = train_one_epoch(model, opt, SynthLoader(), device, epoch=0)
print(json.dumps(report.to_dict(), indent=2, default=str))

In [ ]:
# Persist results (mount Drive first if you want them to survive the session)
# from google.colab import drive; drive.mount('/content/drive')
# out = Path('/content/drive/MyDrive/roadvision_results')
# out.mkdir(parents=True, exist_ok=True)
# (out / f"pilot_{DEVICE}.json").write_text(json.dumps(report.to_dict(), indent=2, default=str))
print("Set DEVICE='cpu' and re-run this notebook in the SAME runtime for Experiment D.")